# 04 — Physics-Informed Feature Engineering

Corrosion is electrochemistry: it needs **(1)** a water film (electrolyte), **(2)** dissolved ions that make the film aggressive (chloride, sulfate, nitrate) and an oxidant, and **(3)** time. These three factors **multiply** — a salty surface that is never wet does not corrode, and a wet surface that is never salty corrodes slowly.

So every feature here is built as a **product of the three factors, integrated over time**:

$$D(t)\;\approx\;\sum_{\text{months}} \underbrace{\text{parking}_m}_{\text{time}}\;\times\;\underbrace{f(\text{wetness}_m)}_{\text{electrolyte gate}}\;\times\;\underbrace{g(\text{Cl}_m,\,\text{SO}_{2,m},\,T_m)}_{\text{aggressiveness}}$$

This mirrors the dose-response form of the **ISO 9223** atmospheric-corrosivity standard (`rate ∝ TOWᵃ·[SO₂]ᵇ·[Cl]ᶜ·e^{f(T)}`). The pipeline is wrapped in functions so the **identical** transforms apply to train and test.

Depends on the labels from [`02_target_construction.ipynb`](02_target_construction.ipynb).

## 0. Parameters & physical constants
Thresholds come from the corrosion literature: a water film forms above **RH ≈ 80%** (ISO 9223 time-of-wetness), kinetics follow **Arrhenius** (≈doubling per +10 °C, `Eₐ ≈ 50 kJ/mol`), and corrosion effectively stops below **0 °C** (frozen electrolyte).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 80)

RH_THRESHOLD   = 80.0     # %RH where a water film begins to form (time-of-wetness gate)
TOW_STEEPNESS  = 5.0      # logistic steepness of the wetness gate around the threshold
EA_J_PER_MOL   = 50_000.0 # Arrhenius activation energy (~doubling per +10 C)
R_GAS          = 8.314    # J/mol/K
T_REF_K        = 298.15   # reference temp (25 C) -> Arrhenius factor = 1 here
ACID_WEIGHT    = 1.0      # relative weight of acid species vs chloride in the corrosivity index
ROLL_WINDOWS   = [6, 12, 24]  # trailing-window months for dose integrals

DATA = Path('..') / 'Data'
OUT  = Path('..') / 'output'; OUT.mkdir(parents=True, exist_ok=True)

## 1. Load the environment files

In [2]:
env_tr = pd.read_csv(DATA / 'environment_training.csv', parse_dates=['month_start_date'])
env_te = pd.read_csv(DATA / 'environment_test.csv',     parse_dates=['month_start_date'])
print('train env:', env_tr.shape, '| test env:', env_te.shape)

train env: (63524, 36) | test env: (14303, 36)


## 2. Monthly mechanism features (instantaneous, per row)
**Factor 1 — is there an electrolyte?** `tow_gate` is a logistic switch around RH 80% (time-of-wetness); `dew_spread = T − T_dew` is small when the skin sweats; `arrhenius_rel` scales kinetics with temperature and is zeroed below freezing.  
**Factor 2 — how aggressive?** total `chloride` (sea salt, the headline pitting agent), `acid_species` (SO₂ + sulphate + HNO₃) and `oxidant` (H₂O₂ + O₃) that converts SO₂→acid in the film.  
**Factor 3 — time:** `parking_hours`.

In [3]:
def add_monthly_features(df):
    df = df.copy()

    # --- Factor 1: electrolyte / time-of-wetness ---
    rh = df['metar_relative_humidity']
    # logistic wetness gate, switched OFF below 0 C: frozen electrolyte -> no ionic circuit.
    # ISO 9223 defines time-of-wetness as hours with RH > 80% AND T > 0 C, so the freeze
    # cutoff applies to the gate itself -> every wet dose built from tow_gate stops when frozen.
    df['tow_gate']    = (1.0 / (1.0 + np.exp(-(rh - RH_THRESHOLD) / TOW_STEEPNESS))) * (df['metar_temperature_c'] >= 0)
    df['rh_excess']   = (rh - RH_THRESHOLD).clip(lower=0)
    df['dew_spread']  = df['metar_temperature_c'] - df['metar_dew_point_c']
    df['condensation']= np.exp(-df['dew_spread'].clip(lower=0) / 3.0)  # ~1 when skin sweats
    df['freezing']    = (df['metar_temperature_c'] < 0).astype(int)

    # Arrhenius factor, relative to 25 C, switched off below 0 C
    Tk = df['temperature']  # Kelvin
    df['arrhenius_rel'] = np.exp(-(EA_J_PER_MOL / R_GAS) * (1.0/Tk - 1.0/T_REF_K))
    df['arrhenius_rel'] = df['arrhenius_rel'] * (df['metar_temperature_c'] >= 0)

    # --- Factor 2: aggressiveness of the electrolyte ---
    df['chloride'] = (df['sea_salt_aerosol_003_05_mixing_ratio']
                      + df['sea_salt_aerosol_05_5_mixing_ratio']
                      + df['sea_salt_aerosol_5_20_mixing_ratio'])
    df['acid_species'] = (df['sulphur_dioxide_mass_mixing_ratio']
                          + df['sulphate_aerosol_mixing_ratio']
                          + df['hno3'])
    df['oxidant'] = df['h2o2'] + df['ozone_mass_mixing_ratio']
    df['black_carbon'] = (df['hydrophilic_black_carbon_aerosol_mixing_ratio']
                          + df['hydrophobic_black_carbon_aerosol_mixing_ratio'])

    # --- Factor 3: time ---
    df['parking_hours'] = df['total_parking_minutes'] / 60.0
    return df

env_tr = add_monthly_features(env_tr)
env_tr[['aircraft_id','year_month','tow_gate','dew_spread','arrhenius_rel','chloride','acid_species','parking_hours']].head()

,aircraft_id,year_month,tow_gate,dew_spread,arrhenius_rel,chloride,acid_species,parking_hours
0,a414b2,2017-04,0.000054,21.159461,1.312683,9.039528e-10,6.895575e-08,687.002500
1,a414b2,2021-07,0.409991,4.632686,1.267281,1.972358e-08,2.443383e-08,476.462222
2,41ef34,2024-01,0.342431,5.023223,0.476026,7.139591e-10,8.677917e-08,497.802778
3,e2d345,2019-05,0.000536,15.785229,1.277693,2.160858e-09,3.895535e-08,302.323056
4,cfa8ab,2025-07,0.575624,3.737505,1.112004,1.437852e-08,2.172774e-08,448.605556


## 3. Monthly dose increments (the multiplicative products)
Each is the per-month contribution to the corrosion integral — **time × wetness × aggressiveness**. `salt_wet_dose` is the most mechanism-faithful term; `acid_prod_dose` encodes that oxidants turn SO₂ into acid; `corrosivity_increment` is the ISO 9223-style combined index (chloride + weighted acid, Arrhenius-scaled, gated by wetness).

In [4]:
def add_dose_increments(df):
    df = df.copy()
    wet_park = df['parking_hours'] * df['tow_gate']          # wet-parking exposure
    df['wet_parking']    = wet_park
    df['salt_wet_dose']  = wet_park * df['chloride']
    df['acid_dose']      = wet_park * df['acid_species']
    df['acid_prod_dose'] = wet_park * df['sulphur_dioxide_mass_mixing_ratio'] * df['oxidant']
    df['arr_wet_dose']   = wet_park * df['arrhenius_rel']
    # ISO 9223-style monthly corrosivity increment (proxy: mixing ratios stand in for deposition)
    df['corrosivity_increment'] = (wet_park * df['arrhenius_rel']
                                   * (df['chloride'] + ACID_WEIGHT * df['acid_species']))
    return df

DOSE_COLS = ['parking_hours','wet_parking','salt_wet_dose','acid_dose',
             'acid_prod_dose','arr_wet_dose','corrosivity_increment']

env_tr = add_dose_increments(env_tr)
env_tr[['aircraft_id','year_month'] + DOSE_COLS].head()

,aircraft_id,year_month,parking_hours,wet_parking,salt_wet_dose,acid_dose,acid_prod_dose,arr_wet_dose,corrosivity_increment
0,a414b2,2017-04,687.002500,0.037335,3.374946e-11,2.574492e-09,1.313691e-16,0.049010,3.423795e-09
1,a414b2,2021-07,476.462222,195.345420,3.852912e-06,4.773037e-06,1.265131e-13,247.557576,1.093150e-05
2,41ef34,2024-01,497.802778,170.463267,1.217038e-07,1.479266e-05,4.875723e-13,81.144905,7.099622e-06
3,e2d345,2019-05,302.323056,0.162159,3.504028e-10,6.316964e-09,4.125941e-16,0.207189,8.518847e-09
4,cfa8ab,2025-07,448.605556,258.228136,3.712939e-06,5.610713e-06,1.461631e-13,287.150668,1.036794e-05


## 4. Cumulative & trailing-window integrals (causal)
Corrosion damage is an **integral of past exposure**, so we accumulate each dose **up to** the current month: lifetime `cumsum` plus trailing 6/12/24-month rolling sums. We also add an **age proxy** (months of history so far) and **environment fingerprints** (long-run means of coastal/industrial markers).

> Age uses *months since first env record* — `environment_test.csv` has no delivery date, so this keeps train and test identical.  
> Windows are row-based (≈months); most aircraft have near-contiguous monthly data.

In [5]:
FINGERPRINT_COLS = {
    'fp_coarse_seasalt': 'sea_salt_aerosol_5_20_mixing_ratio',  # parked near the sea
    'fp_so2':            'sulphur_dioxide_mass_mixing_ratio',   # industrial airport
    'fp_black_carbon':   'black_carbon',                        # urban/soot
    'fp_isoprene':       'isoprene',                            # vegetation / tropical
    'fp_humidity':       'metar_relative_humidity',
    'fp_temp_c':         'metar_temperature_c',
}

def add_cumulative_features(df):
    df = df.sort_values(['aircraft_id', 'month_start_date']).copy()
    g = df.groupby('aircraft_id', sort=False)

    # age proxy + lifetime parking
    df['age_months'] = g.cumcount() + 1

    for col in DOSE_COLS:
        df[f'{col}_cum'] = g[col].cumsum()
        for w in ROLL_WINDOWS:
            df[f'{col}_roll{w}m'] = (g[col]
                .transform(lambda s, w=w: s.rolling(w, min_periods=1).sum()))

    # environment fingerprints (aircraft-level long-run means)
    for name, col in FINGERPRINT_COLS.items():
        df[name] = g[col].transform('mean')
    return df

env_tr = add_cumulative_features(env_tr)
show = ['aircraft_id','year_month','age_months','corrosivity_increment_cum',
        'salt_wet_dose_roll12m','fp_coarse_seasalt']
env_tr[show].head()

,aircraft_id,year_month,age_months,corrosivity_increment_cum,salt_wet_dose_roll12m,fp_coarse_seasalt
63518,002eab,2019-10,1,3.029565e-07,1.958183e-07,1.094853e-09
34301,002eab,2019-11,2,3.044943e-07,1.961087e-07,1.094853e-09
9024,002eab,2019-12,3,3.068613e-07,1.972296e-07,1.094853e-09
18752,002eab,2020-01,4,3.078532e-07,1.981249e-07,1.094853e-09
17062,002eab,2020-02,5,3.080708e-07,1.983053e-07,1.094853e-09


## 5. Wrap the whole pipeline into one function
So train and test get byte-for-byte identical feature engineering.

In [6]:
def build_features(df):
    return add_cumulative_features(add_dose_increments(add_monthly_features(df)))

# (env_tr is already built above; build the test set with the same function)
env_te_feat = build_features(env_te)

ENGINEERED = (['tow_gate','rh_excess','dew_spread','condensation','freezing','arrhenius_rel',
               'chloride','acid_species','oxidant','black_carbon','parking_hours','age_months']
              + [f'{c}_cum' for c in DOSE_COLS]
              + [f'{c}_roll{w}m' for c in DOSE_COLS for w in ROLL_WINDOWS]
              + list(FINGERPRINT_COLS.keys()))
print('engineered feature count:', len(ENGINEERED))
print(ENGINEERED)

engineered feature count: 46
['tow_gate', 'rh_excess', 'dew_spread', 'condensation', 'freezing', 'arrhenius_rel', 'chloride', 'acid_species', 'oxidant', 'black_carbon', 'parking_hours', 'age_months', 'parking_hours_cum', 'wet_parking_cum', 'salt_wet_dose_cum', 'acid_dose_cum', 'acid_prod_dose_cum', 'arr_wet_dose_cum', 'corrosivity_increment_cum', 'parking_hours_roll6m', 'parking_hours_roll12m', 'parking_hours_roll24m', 'wet_parking_roll6m', 'wet_parking_roll12m', 'wet_parking_roll24m', 'salt_wet_dose_roll6m', 'salt_wet_dose_roll12m', 'salt_wet_dose_roll24m', 'acid_dose_roll6m', 'acid_dose_roll12m', 'acid_dose_roll24m', 'acid_prod_dose_roll6m', 'acid_prod_dose_roll12m', 'acid_prod_dose_roll24m', 'arr_wet_dose_roll6m', 'arr_wet_dose_roll12m', 'arr_wet_dose_roll24m', 'corrosivity_increment_roll6m', 'corrosivity_increment_roll12m', 'corrosivity_increment_roll24m', 'fp_coarse_seasalt', 'fp_so2', 'fp_black_carbon', 'fp_isoprene', 'fp_humidity', 'fp_temp_c']


## 6. Build the model-ready TRAINING matrix
Load the labels from notebook 02 and attach features with an **as-of join**: each label row takes the features of its **latest available env month ≤ the reference month**. This keeps the 246 rows whose exact month has no env data (we flag the staleness instead of dropping).

In [7]:
label_files = sorted(OUT.glob('labels_*.csv'))
assert label_files, 'Run notebook 02 first to produce output/labels_*.csv'
labels = pd.read_csv(label_files[-1])
print('using labels:', label_files[-1].name, '|', labels.shape)

labels['ref_ts'] = pd.PeriodIndex(labels['year_month'], freq='M').to_timestamp()
labels = labels.sort_values('ref_ts').reset_index(drop=True)

feat = env_tr.copy()
feat['feat_ts'] = pd.PeriodIndex(feat['year_month'], freq='M').to_timestamp()
feat = feat.rename(columns={'year_month': 'feat_year_month'}).sort_values('feat_ts')

train_mat = pd.merge_asof(labels, feat, left_on='ref_ts', right_on='feat_ts',
                          by='aircraft_id', direction='backward')

# staleness: how many months back the matched features come from (0 = exact month, NaN = no match)
train_mat['feat_gap_months'] = ((train_mat['ref_ts'].dt.year  - train_mat['feat_ts'].dt.year) * 12
                                + (train_mat['ref_ts'].dt.month - train_mat['feat_ts'].dt.month))

print('train matrix:', train_mat.shape)
print('label rows with NO usable features (no prior env at all):',
      int(train_mat['feat_ts'].isna().sum()) if 'feat_ts' in train_mat else 'n/a',
      int(train_mat['corrosivity_increment_cum'].isna().sum()))
train_mat['feat_gap_months'].describe()

using labels: labels_20260611_141930.csv | (1516, 6)


train matrix: (1516, 96)
label rows with NO usable features (no prior env at all): 147 149


count    1369.000000
mean        0.617969
std         3.883839
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        71.000000
Name: feat_gap_months, dtype: float64

In [8]:
# How often did the as-of join fall back to an earlier month, by label?
train_mat['exact_month'] = train_mat['feat_gap_months'] == 0
train_mat.groupby('y')['exact_month'].value_counts().rename('rows')

y  exact_month
0  True           654
   False          104
1  True           616
   False          142
Name: rows, dtype: int64

## 7. Quick signal check
Do the cumulative dose features already separate y=1 from y=0? (They should — the corroded row has ~24 more months of accumulated dose.) Compare medians by label.

In [9]:
check = ['age_months','corrosivity_increment_cum','salt_wet_dose_cum','acid_dose_cum',
         'wet_parking_cum','salt_wet_dose_roll12m','fp_coarse_seasalt','fp_so2']
train_mat.groupby('y')[check].median().T

y,0,1
age_months,4.500000e+01,6.800000e+01
corrosivity_increment_cum,5.353409e-05,8.268445e-05
salt_wet_dose_cum,6.401117e-05,9.888456e-05
acid_dose_cum,3.420025e-05,4.958515e-05
wet_parking_cum,5.372515e+03,8.510621e+03
salt_wet_dose_roll12m,1.620281e-05,1.689347e-05
fp_coarse_seasalt,2.889099e-09,2.895787e-09
fp_so2,4.761585e-09,4.759906e-09


## 8. Persist outputs
Save the model-ready training matrix and the fully-enriched test table (all 14,303 test rows; the specific rows to score get selected later from `sample_submission.csv`).

In [10]:
ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
train_path = OUT / f'train_features_{ts}.csv'
test_path  = OUT / f'test_features_{ts}.csv'
train_mat.to_csv(train_path, index=False)
env_te_feat.to_csv(test_path, index=False)
print('saved:', train_path.name, '|', test_path.name)

saved: train_features_20260611_144825.csv | test_features_20260611_144825.csv


## 9. Feature catalogue (for the report / jury)

| Feature family | Columns | Mechanism |
|---|---|---|
| **Wetness gate** | `tow_gate`, `rh_excess`, `dew_spread`, `condensation`, `freezing` | electrolyte only forms above RH≈80% / small dew spread; off when frozen |
| **Kinetics** | `arrhenius_rel` | reaction rate ~doubles per +10 °C, zero below 0 °C |
| **Aggressiveness** | `chloride`, `acid_species`, `oxidant`, `black_carbon` | pitting (Cl⁻), acidification (SO₂/HNO₃), SO₂→acid conversion, galvanic soot |
| **Dose increments** | `wet_parking`, `salt_wet_dose`, `acid_dose`, `acid_prod_dose`, `arr_wet_dose`, `corrosivity_increment` | monthly time×wetness×aggressiveness products |
| **Cumulative / trailing** | `*_cum`, `*_roll{6,12,24}m` | corrosion = integral of past dose; the y0↔y1 separator |
| **Age** | `age_months` | months of accumulated exposure |
| **Fingerprints** | `fp_*` | long-run coastal/industrial/vegetation base rate of the airport |

The **`corrosivity_increment`** cumulated is our ISO 9223-style corrosivity index — features mirror the standard the industry uses, not ad-hoc columns.

> Next phase: `05_baseline_model` — gradient boosting with **aircraft-grouped CV**, scored on **Brier** vs the 0.25 baseline, plus probability calibration.